In [4]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10  



# Parameters
INPUT_SHAPE = (32, 32, 3)
NUM_CLASSES = 10  

# Define LeNet model
def create_lenet_model():
    model = models.Sequential()
    model.add(layers.Conv2D(6, (5, 5), activation='relu', input_shape=INPUT_SHAPE, padding="same")) 
    # input is of 3,32,32 output will be 6,32,32   but if padding is valid size will be 6,28,28
    model.add(layers.AveragePooling2D(pool_size=(2, 2))) #pooling layer converts input of 6,32,32 to 6,16,16 if same padding, in case of valid padding its 6,14,14
    model.add(layers.Conv2D(16, (5,5), activation='relu')) #the input is 16,16,6  output I-K/1 +1 = 16-5 + 1 = 12,12,16
    model.add(layers.AveragePooling2D(pool_size=(2, 2))) #input is 12,12,16 output is 6,6,16 
    #your input image of 32,32,3 is converted to 6,6,16 = when flattening 6x6x16 = 576
    
    model.add(layers.Flatten()) # size of input is 576 and  there are 576 outputs.
    model.add(layers.Dense(120, activation='relu')) #number of weights in each neuron among 120 neurons = 576 + 1 bias, Total weights and bias = 69120, 120
    model.add(layers.Dense(84, activation='relu')) #No. of weights in each Neuron= 120w + 1, Total Weights and biases = 84X(120w + 1B)  
    #10 classes means 10 neurons Now weights of each neuron depends on number of neurons in previous layer = 84 +1, Total W and B= 840w + 10biases
    model.add(layers.Dense(NUM_CLASSES, activation='softmax')) 
    
    return model


(x_train, y_train), (x_test, y_test) = cifar10.load_data()
print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)
# Preprocessing function
def preprocess(image, label):
    image = tf.image.resize(image, [32, 32])  # Ensure size  
    image = tf.cast(image, tf.float32)/255.0# Normalize to [0,1]   [-1,1]
    label = tf.squeeze(label)  # Remove extra dimension from labels
    return image, label

# Create tf.data Dataset Pipeline For Efficient Loading) 
batch_size = 32 #total images is 50k and batch_Size 32 : no. of batches= 50k/32= 1562.5
train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_ds = train_ds.map(preprocess).shuffle(buffer_size=10000).batch(batch_size).prefetch(tf.data.AUTOTUNE)
#that there will be 32 images in 1 batch each image 32x32x3= 3072  Total pixels in 1 batch = 32X3072= 98304
test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test))
test_ds = test_ds.map(preprocess).batch(batch_size).prefetch(tf.data.AUTOTUNE)

#  Compile Model
model = create_lenet_model()
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',   
              metrics=['accuracy']) # understand accuracy and precision

# Model Summary
model.summary()

# Train Model
model.fit(train_ds, epochs=20, validation_data=test_ds)
#epoch is passing of all training from starting till end in batches defiend by user 

# Evaluate Model
test_loss, test_acc = model.evaluate(test_ds)
print(f'Test accuracy: {test_acc:.4f}')


(50000, 32, 32, 3)
(50000, 1)
(10000, 32, 32, 3)
(10000, 1)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 32, 32, 6)      │           456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_6             │ (None, 16, 16, 6)      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 12, 12, 16)     │         2,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_7             │ (None, 6, 6, 16)       │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 576)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 120)            │        69,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 84)             │        10,164 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 10)             │           850 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 83,126 (324.71 KB)

 Trainable params: 83,126 (324.71 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - accuracy: 0.4108 - loss: 1.6260 - val_accuracy: 0.4851 - val_loss: 1.4655
Epoch 2/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - accuracy: 0.5175 - loss: 1.3518 - val_accuracy: 0.5377 - val_loss: 1.2929
Epoch 3/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.5589 - loss: 1.2404 - val_accuracy: 0.5567 - val_loss: 1.2396
Epoch 4/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.5861 - loss: 1.1578 - val_accuracy: 0.5749 - val_loss: 1.1929
Epoch 5/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.6119 - loss: 1.0921 - val_accuracy: 0.5920 - val_loss: 1.1506
Epoch 6/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.6311 - loss: 1.0329 - val_accuracy: 0.5997 - val_loss: 1.1294
Epoch 7/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - accuracy: 0.6517 - loss: 0.9844 - val_accuracy: 0.6154 - val_loss: 1.1162
Epoch 8/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.6667 - loss: 0.9426